# 🐾 Goofy Screener — Phase 3
### Multi-Market Automated Strategy Screener
**Markets:** 🇺🇸 US  |  🇦🇺 ASX  |  🇯🇵 JPX  
**Strategies:** MA Crossover · RSI · Bollinger Bands · MACD · Mean Reversion  
**Output:** Tiered ranking (S / A / B / Skip) + colour-coded Excel report + charts

---
**How to use this notebook:**
1. Run **Cell 2** to install/check packages
2. Set your market in **Cell 3** (`MARKET = "ALL"` for all three, or `"US"`, `"ASX"`, `"JPX"`)
3. Run **all cells** (Kernel → Restart & Run All)
4. Results appear inline + an Excel file saves to `screener_output/`

> ⏱️ Full run (ALL markets, 117 assets) takes ~8–12 minutes depending on your connection.

## 📦 Cell 1 — Package Check

In [ ]:
# Run this first — installs any missing packages
import subprocess, sys

required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "openpyxl"]
for pkg in required:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg} already installed")
    except ImportError:
        print(f"  📥 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  ✅ {pkg} installed")

print("\n✅ All packages ready!")

## 🔧 Cell 2 — Imports & Setup

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import datetime as dt
import os
import warnings
warnings.filterwarnings("ignore")

# Notebook display settings
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Excel formatting
try:
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
    from openpyxl.utils import get_column_letter
    EXCEL_FORMAT = True
except ImportError:
    EXCEL_FORMAT = False

print("✅ Imports complete")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
print(f"   yfinance {yf.__version__}")

## ⚙️ Cell 3 — Configuration
> **Change `MARKET` here to control which universe runs.**  
> `"ALL"` → all three markets (~117 assets, ~10 min)  
> `"US"` → US large-cap only (~40 assets, ~3 min)  
> `"ASX"` → Australian stocks only (~37 assets, ~3 min)  
> `"JPX"` → Japanese stocks only (~47 assets, ~4 min)

In [ ]:
# ══════════════════════════════════════════════════
#  ▶  CHANGE THIS to control which market runs
# ══════════════════════════════════════════════════
MARKET = "ALL"    # "ALL" | "US" | "ASX" | "JPX"

# ── Date config ──────────────────────────────────
TRAIN_START      = dt.datetime(2016, 1, 1)
TRAIN_END        = dt.datetime(2021, 1, 1)
TEST_END         = dt.datetime.now()
MIN_ROWS         = 400
PERIODS_PER_YEAR = 252

# ── Tier thresholds ───────────────────────────────
TIER_S = {"sharpe": 0.8,  "ret": 30,  "max_dd": -20}   # ⭐ Excellent
TIER_A = {"sharpe": 0.4,  "ret": 10,  "max_dd": -35}   # ✅ Good
TIER_B = {"sharpe": 0.1,  "ret": -10, "max_dd": -50}   # 🔵 Decent
# Below TIER_B → Skip

# ── Output folder ─────────────────────────────────
OUTPUT_DIR = "screener_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TODAY = dt.datetime.now().strftime("%Y-%m-%d")
print(f"✅ Config set | Market: {MARKET} | Run date: {TODAY}")
print(f"   Train: {TRAIN_START.date()} → {TRAIN_END.date()}")
print(f"   Test:  {TRAIN_END.date()} → {TODAY}")

## 🌏 Cell 4 — Asset Universes

In [ ]:
# ── 🇺🇸 US Universe ──────────────────────────────────────────────────────────
US_UNIVERSE = [
    # Tech mega-cap
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSLA", "AMD",
    # Financials
    "JPM", "BAC", "GS", "MS", "V", "MA", "BRK-B",
    # Healthcare
    "JNJ", "UNH", "LLY", "PFE", "ABBV", "MRK",
    # Energy
    "XOM", "CVX", "COP",
    # Consumer
    "PG", "KO", "PEP", "WMT", "COST", "MCD",
    # Industrials
    "CAT", "DE", "BA", "HON", "RTX",
    # ETFs
    "SPY", "QQQ", "GLD", "TLT", "IWM",
]

# ── 🇦🇺 ASX Universe ─────────────────────────────────────────────────────────
ASX_UNIVERSE = [
    # Big 4 Banks
    "CBA.AX", "WBC.AX", "ANZ.AX", "NAB.AX",
    # Resources
    "BHP.AX", "RIO.AX", "FMG.AX", "S32.AX",
    # Energy
    "WDS.AX", "STO.AX", "BPT.AX",
    # Healthcare
    "CSL.AX", "RMD.AX", "COH.AX", "SHL.AX",
    # Retail / Consumer
    "WES.AX", "WOW.AX", "COL.AX", "JBH.AX",
    # Tech
    "XRO.AX", "WTC.AX", "ALU.AX",
    # Infrastructure
    "TCL.AX", "APA.AX", "SKI.AX",
    # REITs
    "GMG.AX", "SCG.AX", "GPT.AX",
    # ETFs
    "IOZ.AX", "STW.AX", "VAS.AX",
]

# ── 🇯🇵 JPX Universe ─────────────────────────────────────────────────────────
JPX_UNIVERSE = [
    # Automotive
    "7203.T", "7267.T", "7261.T", "7272.T", "7269.T",
    # Electronics / Tech
    "6758.T", "6501.T", "6954.T", "6902.T", "6861.T",
    "6762.T", "8035.T", "6857.T", "6723.T", "6594.T",
    # Semiconductors / Chemicals
    "4063.T", "4523.T",
    # Telecom
    "9432.T", "9433.T", "9434.T",
    # Internet / Media
    "9984.T", "4689.T", "6098.T", "4385.T",
    # Gaming
    "7974.T", "9684.T", "7832.T",
    # Banks
    "8306.T", "8316.T", "8411.T",
    # Insurance / Financial
    "8750.T", "8725.T",
    # Consumer / Retail
    "3382.T", "8267.T", "4661.T", "2914.T",
    # Industrials
    "6301.T", "6326.T", "7011.T",
    # Pharma
    "4502.T", "4519.T",
    # Trading Companies
    "8001.T", "8002.T", "8058.T",
    # Transport
    "9022.T", "9020.T",
    # ETFs
    "1321.T", "1306.T",
]

UNIVERSE_MAP = {"US": US_UNIVERSE, "ASX": ASX_UNIVERSE, "JPX": JPX_UNIVERSE}

if MARKET == "ALL":
    MARKETS_TO_RUN = list(UNIVERSE_MAP.keys())
else:
    MARKETS_TO_RUN = [MARKET]

total = sum(len(UNIVERSE_MAP[m]) for m in MARKETS_TO_RUN)
print(f"✅ Universes loaded | Running: {MARKETS_TO_RUN}")
for m in MARKETS_TO_RUN:
    print(f"   {m}: {len(UNIVERSE_MAP[m])} assets")
print(f"   Total: {total} assets")

## 📐 Cell 5 — Strategy Functions & Parameter Grids
All strategies use `.shift(1)` on signals — **zero lookahead bias**.

In [ ]:
def compute_metrics(price, position):
    """Returns Sharpe, Total Return %, Max Drawdown %, Win Rate %"""
    df = pd.DataFrame({"price": price})
    df["pos"]       = position.reindex(df.index).fillna(0)
    df["log_ret"]   = np.log(df["price"] / df["price"].shift(1))
    df["strat_ret"] = df["pos"] * df["log_ret"]
    df["cum"]       = np.exp(df["strat_ret"].cumsum())

    lr = df["strat_ret"].dropna()
    if len(lr) < 10:
        return {"Sharpe": np.nan, "TotalRet": np.nan, "MaxDD": np.nan, "WinRate": np.nan}

    ann    = np.exp(lr.mean() * PERIODS_PER_YEAR) - 1
    vol    = lr.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe = ann / vol if vol != 0 else np.nan
    cum    = df["cum"].dropna()
    mdd    = ((cum - cum.cummax()) / cum.cummax()).min() if len(cum) > 0 else np.nan
    total  = cum.iloc[-1] - 1 if len(cum) > 0 else np.nan
    wins   = (lr > 0).sum() / max(len(lr), 1)

    return {
        "Sharpe":   round(float(sharpe), 3)         if not np.isnan(sharpe) else np.nan,
        "TotalRet": round(float(total) * 100, 2)    if not np.isnan(total)  else np.nan,
        "MaxDD":    round(float(mdd) * 100, 2)      if not np.isnan(mdd)    else np.nan,
        "WinRate":  round(float(wins) * 100, 1)     if not np.isnan(wins)   else np.nan,
    }


# ── Strategy 1: MA Crossover ──────────────────────────────────────────────────
def strategy_ma(p, short=20, long=50):
    return (p.rolling(short).mean() > p.rolling(long).mean()).astype(int).shift(1)

# ── Strategy 2: RSI ───────────────────────────────────────────────────────────
def strategy_rsi(p, period=14, oversold=30, overbought=70):
    d   = p.diff()
    rsi = 100 - (100 / (1 + d.clip(lower=0).rolling(period).mean() /
                            (-d.clip(upper=0)).rolling(period).mean()))
    sig = np.zeros(len(p)); hold = False
    for i, r in enumerate(rsi):
        if np.isnan(r):                  sig[i] = 0
        elif not hold and r < oversold:  hold = True;  sig[i] = 1
        elif hold and r > overbought:    hold = False; sig[i] = 0
        else:                            sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

# ── Strategy 3: Bollinger Bands ───────────────────────────────────────────────
def strategy_bb(p, window=20, num_std=2.0):
    mid = p.rolling(window).mean()
    low = mid - num_std * p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i in range(len(p)):
        pi, m, l = p.iloc[i], mid.iloc[i], low.iloc[i]
        if np.isnan(l):           sig[i] = 0
        elif not hold and pi <= l: hold = True;  sig[i] = 1
        elif hold and pi >= m:     hold = False; sig[i] = 0
        else:                      sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

# ── Strategy 4: MACD ──────────────────────────────────────────────────────────
def strategy_macd(p, fast=12, slow=26, signal_p=9):
    macd = p.ewm(span=fast).mean() - p.ewm(span=slow).mean()
    return (macd > macd.ewm(span=signal_p).mean()).astype(int).shift(1)

# ── Strategy 5: Mean Reversion (Z-Score) ─────────────────────────────────────
def strategy_mr(p, window=20, threshold=1.5):
    z   = (p - p.rolling(window).mean()) / p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i, zi in enumerate(z):
        if np.isnan(zi):                   sig[i] = 0
        elif not hold and zi < -threshold: hold = True;  sig[i] = 1
        elif hold and zi >= 0:             hold = False; sig[i] = 0
        else:                              sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)


STRATEGY_FNS = {
    "MA Crossover":    strategy_ma,
    "RSI":             strategy_rsi,
    "Bollinger Bands": strategy_bb,
    "MACD":            strategy_macd,
    "Mean Reversion":  strategy_mr,
}

STRATEGY_GRIDS = {
    "MA Crossover":    [{"short": s, "long": l}
                        for s in [10, 20, 50] for l in [50, 100, 200] if s < l],
    "RSI":             [{"period": p, "oversold": os, "overbought": ob}
                        for p in [10, 14, 21] for os, ob in [(25,65),(30,70),(35,75)]],
    "Bollinger Bands": [{"window": w, "num_std": s}
                        for w in [10, 20, 30] for s in [1.5, 2.0, 2.5]],
    "MACD":            [{"fast": f, "slow": s, "signal_p": sg}
                        for f in [8, 12] for s in [21, 26] for sg in [7, 9] if f < s],
    "Mean Reversion":  [{"window": w, "threshold": t}
                        for w in [10, 20, 40] for t in [1.0, 1.5, 2.0]],
}

total_combos = sum(len(g) for g in STRATEGY_GRIDS.values())
print(f"✅ 5 strategies loaded | {total_combos} parameter combinations per asset")

## 🏆 Cell 6 — Scoring & Tier System
Each asset is scored 0–100 across four dimensions, then classified into a tier.

In [ ]:
def score_asset(row):
    """
    Composite score (0-100):
      Sharpe       → 40 pts  (quality of risk-adjusted return)
      Total Return → 25 pts  (raw performance)
      Max Drawdown → 20 pts  (capital preservation)
      DD Saved     → 15 pts  (edge over Buy & Hold)
    """
    sharpe  = row.get("OUT Sharpe", np.nan)
    ret     = row.get("OUT Strat Ret %", np.nan)
    mdd     = row.get("OUT Strat Max DD %", np.nan)
    dd_save = row.get("DD Saved %", np.nan)

    if any(pd.isna(v) for v in [sharpe, ret, mdd]):
        return ("Skip", 0.0)

    s_pts  = min(max(sharpe / 1.5, 0), 1) * 40
    r_norm = max(ret + 30, 0) / 130
    r_pts  = min(r_norm, 1) * 25
    d_pts  = min(max((60 + mdd) / 60, 0), 1) * 20
    ds_pts = min(max((dd_save + 20) / 60, 0), 1) * 15 if not pd.isna(dd_save) else 0

    score = round(s_pts + r_pts + d_pts + ds_pts, 1)

    if   sharpe >= TIER_S["sharpe"] and ret >= TIER_S["ret"] and mdd >= TIER_S["max_dd"]: tier = "S"
    elif sharpe >= TIER_A["sharpe"] and ret >= TIER_A["ret"] and mdd >= TIER_A["max_dd"]: tier = "A"
    elif sharpe >= TIER_B["sharpe"]                           and mdd >= TIER_B["max_dd"]: tier = "B"
    else: tier = "Skip"

    return (tier, score)


# Visual explanation of the tier system
tier_df = pd.DataFrame({
    "Tier":       ["⭐ S — Excellent", "✅ A — Good", "🔵 B — Decent", "⬜ Skip"],
    "Min Sharpe": [0.8, 0.4, 0.1, "—"],
    "Min Return": ["30%", "10%", "—", "—"],
    "Max Drawdown": ["-20%", "-35%", "-50%", "worse"],
})
print("\nTier classification system:")
print(tier_df.to_string(index=False))

## 📥 Cell 7 — Download Price Data
Downloads from Yahoo Finance for all selected markets. Takes ~1-3 mins.

In [ ]:
price_data = {}
failed = []

print(f"Downloading data for {MARKETS_TO_RUN}...\n")

for market in MARKETS_TO_RUN:
    assets = UNIVERSE_MAP[market]
    print(f"  ── {market} ({len(assets)} assets) ──")
    for asset in assets:
        try:
            raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                              auto_adjust=True, progress=False)
            if not raw.empty and len(raw) >= MIN_ROWS:
                price_data[asset] = raw["Close"].squeeze()
                print(f"    ✅ {asset:14} {len(raw)} rows")
            else:
                rows = len(raw) if not raw.empty else 0
                print(f"    ⚠️  {asset:14} skipped (only {rows} rows)")
                failed.append(asset)
        except Exception as e:
            print(f"    ❌ {asset:14} error: {e}")
            failed.append(asset)

print(f"\n{'='*50}")
print(f"✅ Downloaded:  {len(price_data)} assets")
print(f"⚠️  Skipped:     {len(failed)} assets")
if failed:
    print(f"   Skipped: {failed}")

## 🚀 Cell 8 — Run Screener
For each asset: grid search all strategies on training data → pick the best → test on out-of-sample data the model never saw.

In [ ]:
all_results = {}  # {market: DataFrame}

for market in MARKETS_TO_RUN:
    assets  = UNIVERSE_MAP[market]
    results = []

    print(f"\n{'='*55}")
    print(f"  Screening {market} market")
    print(f"{'='*55}")

    valid = [a for a in assets if a in price_data]
    print(f"  {len(valid)} valid assets | skipping {len(assets)-len(valid)} failed downloads\n")

    for asset in valid:
        full  = price_data[asset]
        train = full[full.index < TRAIN_END]
        test  = full[full.index >= TRAIN_END]

        if len(train) < 100 or len(test) < 50:
            continue

        # ── Best strategy search (TRAIN period only) ─────────────────────────
        best_sharpe, best_strat, best_params = -999, None, None

        for strat_name, fn in STRATEGY_FNS.items():
            for params in STRATEGY_GRIDS[strat_name]:
                try:
                    pos = fn(train, **params)
                    m   = compute_metrics(train, pos)
                    if not np.isnan(m["Sharpe"]) and m["Sharpe"] > best_sharpe:
                        best_sharpe = m["Sharpe"]
                        best_strat  = strat_name
                        best_params = params
                except:
                    continue

        if best_strat is None:
            continue

        # ── Out-of-sample evaluation (TEST period) ────────────────────────────
        pos_test = STRATEGY_FNS[best_strat](test, **best_params)
        tm       = compute_metrics(test, pos_test)

        # ── Buy & Hold benchmark ──────────────────────────────────────────────
        log     = np.log(test / test.shift(1)).dropna()
        bah_ret = (np.exp(log.cumsum()).iloc[-1] - 1) * 100 if len(log) > 0 else np.nan
        bah_cum = np.exp(log.cumsum()) if len(log) > 0 else pd.Series(dtype=float)
        bah_mdd = ((bah_cum - bah_cum.cummax()) / bah_cum.cummax()).min() * 100 if len(bah_cum) > 0 else np.nan
        dd_saved = round(tm["MaxDD"] - float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan

        row = {
            "Market":             market,
            "Asset":              asset,
            "Best Strategy":      best_strat,
            "Best Params":        str(best_params),
            "Train Sharpe":       round(best_sharpe, 3),
            "OUT Sharpe":         tm["Sharpe"],
            "OUT Win Rate %":     tm["WinRate"],
            "OUT Strat Ret %":    tm["TotalRet"],
            "OUT B&H Ret %":      round(float(bah_ret), 2) if not np.isnan(bah_ret) else np.nan,
            "OUT Strat Max DD %": tm["MaxDD"],
            "OUT B&H Max DD %":   round(float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan,
            "DD Saved %":         dd_saved,
            "Beats B&H":          (tm["TotalRet"] or 0) > (bah_ret or 0),
            "Run Date":           TODAY,
        }

        tier, score = score_asset(row)
        row["Tier"]  = tier
        row["Score"] = score
        results.append(row)

        icon = {"S": "⭐", "A": "✅", "B": "🔵", "Skip": "⬜"}.get(tier, "")
        dd_str = f"{dd_saved:+.1f}%" if not np.isnan(dd_saved) else "N/A"
        print(f"  {icon}[{tier}] {asset:14} → {best_strat:18} | "
              f"Sharpe:{tm['Sharpe']:5.2f} | Ret:{(tm['TotalRet'] or 0):6.0f}% | "
              f"DD:{(tm['MaxDD'] or 0):5.1f}% | DDsaved:{dd_str:>7} | Score:{score:.0f}")

    all_results[market] = pd.DataFrame(results)

print(f"\n{'='*55}")
print("✅ Screener complete!")
for m, df in all_results.items():
    print(f"   {m}: {len(df)} assets processed")

## 📊 Cell 9 — Results Summary

In [ ]:
# Combine all markets
combined = pd.concat(all_results.values(), ignore_index=True) if all_results else pd.DataFrame()

if not combined.empty:
    total     = len(combined)
    beats_bah = combined["Beats B&H"].sum()
    tier_counts = combined["Tier"].value_counts()

    print(f"{'='*55}")
    print(f"  GOOFY SCREENER — PHASE 3 SUMMARY  ({TODAY})")
    print(f"{'='*55}")
    print(f"  Total assets screened : {total}")
    print(f"  Beat Buy & Hold       : {beats_bah} / {total} ({beats_bah/total*100:.1f}%)")
    print(f"\n  Tier breakdown:")
    for tier in ["S", "A", "B", "Skip"]:
        icon = {"S":"⭐","A":"✅","B":"🔵","Skip":"⬜"}.get(tier,"")
        count = tier_counts.get(tier, 0)
        bar = "█" * count
        print(f"    {icon} {tier:5} : {count:3}  {bar}")

    print(f"\n  Strategy distribution:")
    for strat, count in combined["Best Strategy"].value_counts().items():
        bar = "█" * count
        print(f"    {strat:20}: {count:3}  {bar}")
    print(f"{'='*55}")

## 🥇 Cell 10 — Top Performers (Styled Table)

In [ ]:
if not combined.empty:
    display_cols = ["Market", "Asset", "Best Strategy", "Tier", "Score",
                    "OUT Sharpe", "OUT Strat Ret %", "OUT B&H Ret %",
                    "OUT Strat Max DD %", "OUT B&H Max DD %", "DD Saved %", "Beats B&H"]

    top = combined.sort_values("Score", ascending=False).head(20)[display_cols].reset_index(drop=True)

    def color_tier(val):
        colors = {"S": "background-color: #FFD700; font-weight: bold",
                  "A": "background-color: #90EE90; font-weight: bold",
                  "B": "background-color: #87CEEB; font-weight: bold",
                  "Skip": "background-color: #D3D3D3"}
        return colors.get(val, "")

    def color_sharpe(val):
        try:
            v = float(val)
            if v >= 0.8: return "background-color: #D5F5E3"
            if v >= 0.4: return "background-color: #FDFEFE"
            if v < 0:    return "background-color: #FADBD8"
        except: pass
        return ""

    def color_beats(val):
        return "background-color: #D5F5E3" if val else "background-color: #FADBD8"

    def color_dd_saved(val):
        try:
            v = float(val)
            if v >= 10: return "background-color: #D5F5E3"
            if v < 0:   return "background-color: #FADBD8"
        except: pass
        return ""

    styled = (top.style
        .applymap(color_tier, subset=["Tier"])
        .applymap(color_sharpe, subset=["OUT Sharpe"])
        .applymap(color_beats, subset=["Beats B&H"])
        .applymap(color_dd_saved, subset=["DD Saved %"])
        .format({"OUT Sharpe": "{:.3f}", "Score": "{:.0f}",
                 "OUT Strat Ret %": "{:.1f}%", "OUT B&H Ret %": "{:.1f}%",
                 "OUT Strat Max DD %": "{:.1f}%", "OUT B&H Max DD %": "{:.1f}%",
                 "DD Saved %": "{:.1f}%"})
        .set_caption(f"Top 20 Assets by Score — Run: {TODAY}"))

    display(styled)

## 📈 Cell 11 — Charts & Visualisations

In [ ]:
if not combined.empty:
    fig = plt.figure(figsize=(18, 14))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    STRAT_COLORS_MAP = {
        "MA Crossover":   "#AED6F1",
        "RSI":            "#FAD7A0",
        "Bollinger Bands":"#A9DFBF",
        "MACD":           "#D7BDE2",
        "Mean Reversion": "#F1948A",
    }
    TIER_COLOR_MAP = {"S": "#FFD700", "A": "#90EE90", "B": "#87CEEB", "Skip": "#D3D3D3"}

    # ── Chart 1: Top 15 by Sharpe (horizontal bar) ───────────────────────────
    ax1 = fig.add_subplot(gs[0, :2])
    top15 = combined.nlargest(15, "OUT Sharpe")
    colors_bar = [STRAT_COLORS_MAP.get(s, "#CCCCCC") for s in top15["Best Strategy"]]
    bars = ax1.barh(top15["Asset"] + " (" + top15["Market"] + ")",
                    top15["OUT Sharpe"], color=colors_bar, edgecolor="white", linewidth=0.5)
    ax1.axvline(x=0.8, color="red", linestyle="--", alpha=0.5, label="Tier S threshold (0.8)")
    ax1.axvline(x=0.4, color="orange", linestyle="--", alpha=0.5, label="Tier A threshold (0.4)")
    ax1.set_xlabel("Out-of-Sample Sharpe Ratio")
    ax1.set_title("Top 15 Assets by OOS Sharpe", fontweight="bold", fontsize=12)
    ax1.legend(fontsize=8)
    ax1.invert_yaxis()

    # ── Chart 2: Tier breakdown ────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    tier_order = ["S", "A", "B", "Skip"]
    tier_vals  = [combined["Tier"].value_counts().get(t, 0) for t in tier_order]
    tier_clrs  = [TIER_COLOR_MAP[t] for t in tier_order]
    wedges, texts, autotexts = ax2.pie(
        tier_vals, labels=[f"{t}\n({v})" for t, v in zip(tier_order, tier_vals)],
        colors=tier_clrs, autopct="%1.0f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 2})
    for at in autotexts: at.set_fontsize(9)
    ax2.set_title("Tier Distribution", fontweight="bold", fontsize=12)

    # ── Chart 3: Strategy distribution ────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, :2])
    strat_counts = combined["Best Strategy"].value_counts()
    bar_colors   = [STRAT_COLORS_MAP.get(s, "#CCCCCC") for s in strat_counts.index]
    bars3 = ax3.bar(strat_counts.index, strat_counts.values,
                    color=bar_colors, edgecolor="white", linewidth=0.8)
    ax3.bar_label(bars3, padding=3, fontsize=10)
    ax3.set_ylabel("Number of Assets")
    ax3.set_title("Strategy Distribution — Best Strategy Wins", fontweight="bold", fontsize=12)
    ax3.set_ylim(0, strat_counts.max() * 1.15)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=15, ha="right")

    # ── Chart 4: Sharpe distribution by market ────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 2])
    market_colors = {"US": "#3498DB", "ASX": "#E67E22", "JPX": "#E74C3C"}
    for mkt in combined["Market"].unique():
        subset = combined[combined["Market"] == mkt]["OUT Sharpe"].dropna()
        ax4.hist(subset, bins=12, alpha=0.6, label=mkt,
                 color=market_colors.get(mkt, "#999"), edgecolor="white")
    ax4.axvline(x=0.8, color="red", linestyle="--", alpha=0.6)
    ax4.axvline(x=0.4, color="orange", linestyle="--", alpha=0.6)
    ax4.set_xlabel("OOS Sharpe Ratio")
    ax4.set_ylabel("Count")
    ax4.set_title("Sharpe Distribution by Market", fontweight="bold", fontsize=12)
    ax4.legend()

    # ── Chart 5: Strategy DD Saved scatter ────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, :2])
    colors_scatter = [STRAT_COLORS_MAP.get(s, "#CCCCCC") for s in combined["Best Strategy"]]
    sc = ax5.scatter(combined["OUT Sharpe"], combined["DD Saved %"],
                     c=colors_scatter, alpha=0.7, s=50, edgecolors="white", linewidth=0.3)
    ax5.axhline(y=0, color="black", linestyle="-", alpha=0.3)
    ax5.axvline(x=0.8, color="red", linestyle="--", alpha=0.4)
    ax5.set_xlabel("Out-of-Sample Sharpe Ratio")
    ax5.set_ylabel("Drawdown Saved vs B&H (%)")
    ax5.set_title("Risk-Adjusted Return vs Drawdown Protection", fontweight="bold", fontsize=12)
    # Legend for strategies
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=STRAT_COLORS_MAP[s], label=s) for s in STRAT_COLORS_MAP]
    ax5.legend(handles=legend_elements, fontsize=7, loc="lower right")

    # ── Chart 6: Beats B&H by market ──────────────────────────────────────────
    ax6 = fig.add_subplot(gs[2, 2])
    market_list = sorted(combined["Market"].unique())
    beats_pct   = [combined[combined["Market"]==m]["Beats B&H"].mean()*100 for m in market_list]
    bar_c       = [market_colors.get(m, "#999") for m in market_list]
    bars6 = ax6.bar(market_list, beats_pct, color=bar_c, edgecolor="white")
    ax6.bar_label(bars6, fmt="%.0f%%", padding=3, fontsize=10)
    ax6.set_ylabel("% of Assets Beating B&H")
    ax6.set_title("% Beating Buy & Hold by Market", fontweight="bold", fontsize=12)
    ax6.set_ylim(0, max(beats_pct) * 1.25 if beats_pct else 50)

    fig.suptitle(f"Goofy Screener Phase 3 — {TODAY}", fontsize=15, fontweight="bold", y=0.98)
    plt.savefig(os.path.join(OUTPUT_DIR, f"screener_charts_{TODAY}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n✅ Charts saved to screener_output/screener_charts_{TODAY}.png")

## 📉 Cell 12 — Equity Curves (Top 6 Assets)

In [ ]:
if not combined.empty:
    top6 = combined.nlargest(6, "OUT Sharpe")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, (_, row) in enumerate(top6.iterrows()):
        asset  = row["Asset"]
        strat  = row["Best Strategy"]
        params_str = row.get("Best Params", "{}")

        if asset not in price_data:
            continue

        # Reconstruct best params
        import ast
        try:
            bp = ast.literal_eval(params_str)
        except:
            continue

        full  = price_data[asset]
        test  = full[full.index >= TRAIN_END]
        if len(test) < 50:
            continue

        pos      = STRATEGY_FNS[strat](test, **bp)
        log      = np.log(test / test.shift(1)).dropna()
        strat_eq = np.exp((pos.reindex(log.index).fillna(0) * log).cumsum())
        bah_eq   = np.exp(log.cumsum())

        ax = axes[i]
        ax.plot(strat_eq.index, strat_eq.values, label=f"Strategy ({strat})",
                color="#2ECC71", linewidth=1.5)
        ax.plot(bah_eq.index, bah_eq.values, label="Buy & Hold",
                color="#E74C3C", linewidth=1.2, linestyle="--", alpha=0.8)
        ax.fill_between(strat_eq.index, strat_eq.values, 1, alpha=0.08, color="#2ECC71")
        ax.axhline(y=1, color="black", linestyle="-", alpha=0.2, linewidth=0.8)
        ax.set_title(f"{asset}  |  {strat}\nSharpe: {row['OUT Sharpe']:.3f}  |  "
                     f"Tier: {row['Tier']}  |  DD Saved: {row['DD Saved %']:.1f}%",
                     fontsize=9, fontweight="bold")
        ax.set_ylabel("Portfolio Value (1.0 = start)")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"Equity Curves — Top 6 Assets by Sharpe (OOS: 2021–{TODAY[:4]})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"equity_curves_{TODAY}.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n✅ Equity curves saved to screener_output/equity_curves_{TODAY}.png")

## 🌐 Cell 13 — Per-Market Breakdown

In [ ]:
if not combined.empty:
    market_flags = {"US": "🇺🇸", "ASX": "🇦🇺", "JPX": "🇯🇵"}
    display_cols = ["Asset", "Best Strategy", "Tier", "Score", "OUT Sharpe",
                    "OUT Strat Ret %", "OUT B&H Ret %", "OUT Strat Max DD %",
                    "DD Saved %", "Beats B&H"]

    def color_tier(val):
        return {"S": "background-color: #FFD700; font-weight: bold",
                "A": "background-color: #90EE90; font-weight: bold",
                "B": "background-color: #87CEEB; font-weight: bold",
                "Skip": "background-color: #D3D3D3"}.get(val, "")

    def color_sharpe(val):
        try:
            v = float(val)
            if v >= 0.8: return "background-color: #D5F5E3"
            if v < 0:    return "background-color: #FADBD8"
        except: pass
        return ""

    def color_beats(val):
        return "background-color: #D5F5E3" if val else "background-color: #FADBD8"

    for market in MARKETS_TO_RUN:
        if market not in all_results or all_results[market].empty:
            continue
        flag = market_flags.get(market, "")
        df_m = all_results[market].sort_values("Score", ascending=False)[display_cols].reset_index(drop=True)

        s_count = len(df_m[df_m["Tier"] == "S"])
        a_count = len(df_m[df_m["Tier"] == "A"])
        beats   = df_m["Beats B&H"].sum()

        print(f"\n{flag} {market} — {len(df_m)} assets | ⭐{s_count} S-tier | ✅{a_count} A-tier | {beats} beat B&H")

        styled = (df_m.style
            .applymap(color_tier, subset=["Tier"])
            .applymap(color_sharpe, subset=["OUT Sharpe"])
            .applymap(color_beats, subset=["Beats B&H"])
            .format({"OUT Sharpe": "{:.3f}", "Score": "{:.0f}",
                     "OUT Strat Ret %": "{:.1f}%", "OUT B&H Ret %": "{:.1f}%",
                     "OUT Strat Max DD %": "{:.1f}%", "DD Saved %": "{:.1f}%"})
            .set_caption(f"{flag} {market} Results — sorted by Score"))
        display(styled)

## 💾 Cell 14 — Export to Excel
Saves a colour-coded multi-tab Excel report to the `screener_output/` folder.

In [ ]:
def write_excel_report(all_results, today):
    from openpyxl import Workbook
    fname  = f"Goofy_Phase3_{today}.xlsx"
    path   = os.path.join(OUTPUT_DIR, fname)
    wb     = Workbook()
    wb.remove(wb.active)

    STRAT_COLORS_XLSX = {
        "MA Crossover":    "AED6F1",
        "RSI":             "FAD7A0",
        "Bollinger Bands": "A9DFBF",
        "MACD":            "D7BDE2",
        "Mean Reversion":  "F1948A",
    }
    TIER_COLORS_XLSX = {"S": "FFD700", "A": "90EE90", "B": "87CEEB", "Skip": "D3D3D3"}
    market_flags = {"US": "🇺🇸", "ASX": "🇦🇺", "JPX": "🇯🇵"}

    all_dfs = []
    cols_to_show = ["Market", "Asset", "Best Strategy", "Train Sharpe", "OUT Sharpe",
                    "OUT Win Rate %", "OUT Strat Ret %", "OUT B&H Ret %",
                    "OUT Strat Max DD %", "OUT B&H Max DD %", "DD Saved %",
                    "Beats B&H", "Run Date", "Tier", "Score"]

    def fmt_sheet(ws, df_in):
        if not EXCEL_FORMAT: return
        THIN = Side(style="thin", color="CCCCCC")
        BRD  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
        HDR  = PatternFill("solid", fgColor="1C2833")
        GRN  = PatternFill("solid", fgColor="D5F5E3")
        RED  = PatternFill("solid", fgColor="FADBD8")
        headers  = [c.value for c in ws[1]]
        col_idx  = {h: i+1 for i, h in enumerate(headers)}
        for cell in ws[1]:
            cell.fill = HDR; cell.font = Font(bold=True, color="FFFFFF", size=10)
            cell.alignment = Alignment(horizontal="center"); cell.border = BRD
        ws.row_dimensions[1].height = 28
        for row in ws.iter_rows(min_row=2):
            sv = row[col_idx.get("Best Strategy",1)-1].value
            tv = row[col_idx.get("Tier",1)-1].value
            bv = row[col_idx.get("Beats B&H",1)-1].value
            for cell in row:
                cell.border = BRD
                cell.alignment = Alignment(horizontal="center"); cell.font = Font(size=10)
                h = headers[cell.column-1] if cell.column <= len(headers) else ""
                if h == "Best Strategy" and sv:
                    cell.fill = PatternFill("solid", fgColor=STRAT_COLORS_XLSX.get(sv,"FFFFFF"))
                    cell.font = Font(bold=True, size=10)
                elif h == "Tier" and tv:
                    cell.fill = PatternFill("solid", fgColor=TIER_COLORS_XLSX.get(tv,"FFFFFF"))
                    cell.font = Font(bold=True, size=10)
                elif h == "OUT Sharpe":
                    try:
                        v = float(cell.value or 0)
                        if v >= 0.8: cell.fill = GRN
                        elif v < 0:  cell.fill = RED
                    except: pass
                elif h == "DD Saved %":
                    try:
                        v = float(cell.value or 0)
                        if v >= 10: cell.fill = GRN
                        elif v < 0: cell.fill = RED
                    except: pass
                elif h == "Beats B&H":
                    cell.fill = GRN if bv else RED
        for col in ws.columns:
            w = max((len(str(c.value or "")) for c in col), default=0)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(w+4,32)
        ws.freeze_panes = "A2"

    # Per-market tabs
    for market, df in all_results.items():
        if df.empty: continue
        df_s  = df.sort_values("Score", ascending=False).reset_index(drop=True)
        avail = [c for c in cols_to_show if c in df_s.columns]
        flag  = market_flags.get(market, "")
        ws    = wb.create_sheet(title=f"{flag} {market}")
        ws.append(avail)
        for _, row in df_s[avail].iterrows():
            ws.append([row[c] for c in avail])
        fmt_sheet(ws, df_s[avail])
        all_dfs.append(df_s)

    # Top performers tab
    if all_dfs:
        combined_all = pd.concat(all_dfs, ignore_index=True)
        ws_top = wb.create_sheet("⭐ Top Performers", index=0)
        ws_top.append(["GOOFY SCREENER — PHASE 3 RESULTS"])
        ws_top.append([f"Run: {today}  |  Markets: {'+'.join(all_results.keys())}  |  Assets: {len(combined_all)}"])
        ws_top.append([])
        ws_top.append(["TIER", "CRITERIA"])
        ws_top.append(["S (⭐ Excellent)", "Sharpe ≥ 0.8, Return ≥ 30%, Max DD ≥ -20%"])
        ws_top.append(["A (✅ Good)",       "Sharpe ≥ 0.4, Return ≥ 10%, Max DD ≥ -35%"])
        ws_top.append(["B (🔵 Decent)",    "Sharpe ≥ 0.1, Max DD ≥ -50%"])
        ws_top.append(["Skip (⬜)",         "Below all thresholds"])
        ws_top.append([])
        ws_top.append(["TIER BREAKDOWN"])
        for tier in ["S","A","B","Skip"]:
            ws_top.append([tier, len(combined_all[combined_all["Tier"]==tier])])
        ws_top.append([])
        sa = combined_all[combined_all["Tier"].isin(["S","A"])].sort_values("Score",ascending=False)
        sa_cols = ["Market","Asset","Best Strategy","Tier","Score","OUT Sharpe",
                   "OUT Strat Ret %","OUT B&H Ret %","OUT Strat Max DD %","DD Saved %","Beats B&H"]
        ws_top.append(["── S & A TIER (Best Opportunities) ──"])
        ws_top.append([c for c in sa_cols if c in sa.columns])
        for _, row in sa[[c for c in sa_cols if c in sa.columns]].iterrows():
            ws_top.append([row[c] for c in sa_cols if c in sa.columns])
        if EXCEL_FORMAT:
            ws_top["A1"].font = Font(bold=True, size=14)
            ws_top["A2"].font = Font(italic=True, size=10)

        # Strategy distribution tab
        ws_dist = wb.create_sheet("📊 Strategy Distribution")
        strat_dist = combined_all.groupby("Best Strategy").apply(
            lambda x: pd.Series({
                "US Count":  len(x[x["Market"]=="US"]),
                "ASX Count": len(x[x["Market"]=="ASX"]),
                "JPX Count": len(x[x["Market"]=="JPX"]),
                "Total":     len(x)
            })
        ).reset_index()
        ws_dist.append(["Strategy","US Count","ASX Count","JPX Count","Total"])
        for _, row in strat_dist.iterrows():
            ws_dist.append([row["Best Strategy"],
                            int(row["US Count"]),int(row["ASX Count"]),
                            int(row["JPX Count"]),int(row["Total"])])

    wb.save(path)
    return path


# Run export
if not combined.empty:
    excel_path = write_excel_report(all_results, TODAY)
    print(f"\n✅ Excel report saved!")
    print(f"   📁 {excel_path}")
    print(f"\n   Open the file from your screener_output/ folder in Excel or Numbers.")
else:
    print("⚠️  No results to export — check if data downloaded correctly.")

## 🔁 Cell 15 — Quick Re-run (Single Asset)
Use this cell to quickly test or inspect a single asset without re-running everything.

In [ ]:
# ── Change these two lines ──────────────────────────────────────────
INSPECT_ASSET    = "8725.T"    # any asset that was screened
INSPECT_STRATEGY = "RSI"       # any strategy name
# ────────────────────────────────────────────────────────────────────

if INSPECT_ASSET in price_data:
    full  = price_data[INSPECT_ASSET]
    train = full[full.index < TRAIN_END]
    test  = full[full.index >= TRAIN_END]

    # Find best params on train
    best_sharpe, best_params = -999, None
    for params in STRATEGY_GRIDS[INSPECT_STRATEGY]:
        try:
            pos = STRATEGY_FNS[INSPECT_STRATEGY](train, **params)
            m   = compute_metrics(train, pos)
            if not np.isnan(m["Sharpe"]) and m["Sharpe"] > best_sharpe:
                best_sharpe = m["Sharpe"]
                best_params = params
        except: continue

    print(f"Asset: {INSPECT_ASSET} | Strategy: {INSPECT_STRATEGY}")
    print(f"Best params (train): {best_params}  |  Train Sharpe: {best_sharpe:.3f}")

    pos_test = STRATEGY_FNS[INSPECT_STRATEGY](test, **best_params)
    tm = compute_metrics(test, pos_test)

    log     = np.log(test / test.shift(1)).dropna()
    bah_ret = (np.exp(log.cumsum()).iloc[-1] - 1) * 100
    bah_cum = np.exp(log.cumsum())
    bah_mdd = ((bah_cum - bah_cum.cummax()) / bah_cum.cummax()).min() * 100

    print(f"\n OUT-OF-SAMPLE RESULTS")
    print(f"  Sharpe:       {tm['Sharpe']:.3f}")
    print(f"  Total Return: {tm['TotalRet']:.1f}%  vs  B&H: {bah_ret:.1f}%")
    print(f"  Max Drawdown: {tm['MaxDD']:.1f}%  vs  B&H: {bah_mdd:.1f}%")
    print(f"  DD Saved:     {tm['MaxDD'] - bah_mdd:+.1f}%")
    print(f"  Win Rate:     {tm['WinRate']:.1f}%")

    # Plot
    strat_eq = np.exp((pos_test.reindex(log.index).fillna(0) * log).cumsum())
    bah_eq   = np.exp(log.cumsum())

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(strat_eq.index, strat_eq.values, label=f"{INSPECT_STRATEGY}", color="#2ECC71", linewidth=2)
    ax.plot(bah_eq.index, bah_eq.values, label="Buy & Hold", color="#E74C3C",
            linewidth=1.5, linestyle="--")
    ax.fill_between(strat_eq.index, strat_eq.values, 1, alpha=0.1, color="#2ECC71")
    ax.axhline(y=1, color="black", linestyle="-", alpha=0.2)
    ax.set_title(f"{INSPECT_ASSET} — {INSPECT_STRATEGY} (OOS equity curve)", fontweight="bold")
    ax.set_ylabel("Portfolio Value"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print(f"⚠️  {INSPECT_ASSET} not in downloaded data. Check spelling or run Cell 7 first.")